# Development of Arctic boundary current model

See `Development/TestingNavierStokes??.ipynb` series for development of NS solver in rectangular domain.

0. `ArcticBoundaryCurrentModel0.ipynb`: Adapt wind forced dishpan steady NS solver to Arctic boundary current geometry.
1. `ArcticBoundaryCurrentModel1.ipynb`: More work on gmsh domain. 
2. `ArcticBoundaryCurrentModel2.ipynb`: More work on gmsh domain. Add cooling and freshening patches to cylinder.
3. `ArcticBoundaryCurrentModel3.ipynb`: More work on gmsh domain. Add warming/salinification patch. But it still errors when defining the 3D domain (it only works on the surface of the gmsh domain).
4.  `ArcticBoundaryCurrentModel4.ipynb`: More work on gmsh domain. Try to get cylindrical disk to work. This succeeds.
5.  `ArcticBoundaryCurrentModel5.ipynb`: More work on gmsh domain. Try to add cutoff cone beneath cylinder. This succeeds, except the gridap solution is zero in the cone.
6.  `ArcticBoundaryCurrentModel6.ipynb`: More work on gmsh domain. Try to join cylinder and cone beneath cylinder. See also `GridapGmsh_demo.ipynb`. This succeeds.
7.  `ArcticBoundaryCurrentModel7.ipynb`: More work on gmsh domain. Try to add freshening and cooling patches...this takes a long time to get the Gmsh .geo file working right. See Development/mwe_openCASCADE_{errors,works}.geo.
8.  `ArcticBoundaryCurrentModel8.ipynb`: More work on gmsh domain. Try to add warming/salinification patch. This succeeds.
9.  `ArcticBoundaryCurrentModel9.ipynb`: Experiment with simple forcing fields, parameters, and mesh options. ~34000 tetrahedra takes about 6.5 minutes to run for an O(1m) size domain and an Ekman depth of 0.2m and a uniform wind stress.
10.  `ArcticBoundaryCurrentModel10.ipynb`: Switch to azimuthal windstress with ~18k tetraheda (takes 2.5mins). Explore decreasing viscosity and boundary mesh only. The pressure field is very weak in 3D with large viscosity.

twnh July '25

## Problem statement

The ultimate goal is to solve a nonlinear multi-field PDE. Consider here the lid-stress-driven flow for the incompressible rotating Navier-Stokes equations. Formally, the PDE we want to solve is: find the velocity vector $u$ and the pressure anomaly $p$ such that

$$
\left\lbrace
\begin{aligned}
-\nu  \nabla^2 u +  (u \nabla) u + (1/\rho_0) \nabla p + f  \hat{\mathbf k} \times u = 0 &\text{ in }\Omega,\\
\nabla\cdot u = 0 &\text{ in } \Omega,\\
\boldsymbol{t} \cdot \boldsymbol{\sigma} \cdot \mathbf{n} = \tau_{\text{imposed}} &\text{ on } \Gamma_s, \\
u = 0 &\text{ on } \Gamma_w,
\end{aligned}
\right.
$$

where the computational domain is the rectangle $\Omega \doteq (0,L_x) \times (-L_y/2,L_y/2) \times (-L_z,0)$, ${\mathbf n}$ is the unit outward normal, and $\boldsymbol{t}$ is the tangential direction at the surface, $\Gamma_s$. The driving force is the tangential stress $\tau_{\text{imposed}}$ (units of $\text{m}^{2} \text{s}^{-2}$ ). The mean value of the pressure anomaly is constrained to equal zero,

$$
\int_\Omega p \ {\rm d}\Omega = 0 .
$$


In [51]:
using Gridap
using GridapSolvers
using GridapSolvers.LinearSolvers, GridapSolvers.MultilevelTools, GridapSolvers.NonlinearSolvers
using GridapSolvers.BlockSolvers: LinearSystemBlock, NonlinearSystemBlock, BiformBlock, BlockTriangularSolver
using Gridap.MultiField
using GridapGmsh
using LinearAlgebra

#### Define physical parameters

In [52]:
# Physical properties
# ν = 1e-6              # (m^2/s) kinematic viscosity
ν = 5.0e-3              # (m^2/s) kinematic viscosity
ρₒ = 1000.0             # (kg/m^3) reference density
rotation_period = 6.0   # (s) rotation rate
f = 2*2*π/rotation_period
Ekman_layer_depth = sqrt(2*ν / f) # (m), Ekman layer depth
println("Ekman layer depth: ", Ekman_layer_depth, " m")

# Surface stress boundary condition
# u₁₀(y) = 1.0     # m s⁻¹, average wind velocity 10 meters above the ocean
# u₁₀(y) =  1.0 .* cos(π.*y./Ly)     # m s⁻¹, average wind velocity 10 meters above the ocean
function u₁₀(x_in)
    x,y = x_in[1], x_in[2]
    r = sqrt(x^2 + y^2)
    θ = atan(y, x)
    radial_wind = 1.0 * r
    u₁₀ = radial_wind * [ - sin(θ),  cos(θ), 0.0 ] # m s⁻¹
    return u₁₀
end

cᴰ = 2.5e-3 # dimensionless drag coefficient
ρₐ = 1.225  # kg m⁻³, average density of air at sea-level
Qᵘ(x) = VectorValue((ρₐ / ρₒ) * cᴰ * u₁₀(x) * norm(u₁₀(x))) # m² s⁻²

Ekman layer depth: 0.0690988298942671 m


Qᵘ (generic function with 1 method)

Load model geometry.

In [53]:
model = GmshDiscreteModel("ArcticBasin10.msh")
labels = get_face_labeling(model)
label_names = model.face_labeling.tag_to_name

Info    : Reading 'ArcticBasin10.msh'...
Info    : 32 entities
Info    : 1827 nodes
Info    : 3650 elements
Info    : Done reading 'ArcticBasin10.msh'


7-element Vector{String}:
 "Slope"
 "WarmingPatch"
 "Wall"
 "Bottom"
 "FresheningPatch"
 "CoolingPatch"
 "ArcticBasin"

## FE spaces

See: `Development/TestingNavierStokes??.ipynb` for more info on these spaces

In [54]:
order = 2
qdegree = 2*(order+1)
reffeᵤ = ReferenceFE(lagrangian,VectorValue{3,Float64},order)
V = TestFESpace(model,reffeᵤ,conformity=:H1,labels=labels,dirichlet_tags=["Wall","Bottom","Slope"])
reffeₚ = ReferenceFE(lagrangian,Float64,order-1;space=:P)
Q = TestFESpace(model,reffeₚ,conformity=:L2,constraint=:zeromean)
uD0 = VectorValue(0,0,0)            # Velocity vanishes on the boundaries except the top
U = TrialFESpace(V,uD0)
P = TrialFESpace(Q)
mfs = Gridap.MultiField.BlockMultiFieldStyle()
Y = MultiFieldFESpace([V, Q];style=mfs)
X = MultiFieldFESpace([U, P];style=mfs)

MultiFieldFESpace()

## Triangulation and integration quadrature

From the discrete model we can define the triangulation and integration measure

In [55]:
degree = order
Ω = Triangulation(model)
dΩ = Measure(Ω,qdegree)
Γ = BoundaryTriangulation(model,tags="CoolingPatch")
dΓ = Measure(Γ,degree)

GenericMeasure()

## Steady nonlinear Navier-Stokes solver with Coriolis force.

Use a preconditioned FGMRES solver. See: https://gridap.github.io/GridapSolvers.jl/stable/Examples/NavierStokes/

In [56]:
khat = VectorValue(0,0,1)
Coriolis(u,v,dΩ) = ∫(f * cross(khat, u) ⋅ v)dΩ # Coriolis term
b((v,q),dΓ) =  ∫(- v ⋅ Qᵘ)dΓ    # Boundary condition for the velocity

α = 1.e2    # Stabilization parameter
Π_Qh = LocalProjectionMap(divergence,Q,qdegree)
graddiv(u,v,dΩ) = ∫(α*(∇⋅v)⋅Π_Qh(u))dΩ

conv(u,∇u) = (∇u')⋅u
dconv(du,∇du,u,∇u) = conv(u,∇du)+conv(du,∇u)
c(u,v,dΩ) = ∫(v⊙(conv∘(u,∇(u))))dΩ
dc(u,du,dv,dΩ) = ∫(dv⊙(dconv∘(du,∇(du),u,∇(u))))dΩ

lap(u,v,dΩ) = ∫(ν*∇(v)⊙∇(u))dΩ

jac_u(u,du,dv,dΩ) = lap(du,dv,dΩ) + dc(u,du,dv,dΩ) + graddiv(du,dv,dΩ)
jac_u(u,du,dv,dΩ) = lap(du,dv,dΩ) + graddiv(du,dv,dΩ) + Coriolis(du,dv,dΩ)
jac((u,p),(du,dp),(dv,dq),dΩ) = jac_u(u,du,dv,dΩ) - ∫(divergence(dv)*dp)dΩ - ∫(divergence(du)*dq)dΩ

res_u(u,v,dΩ) = lap(u,v,dΩ) + c(u,v,dΩ) + graddiv(u,v,dΩ)
res_u(u,v,dΩ) = lap(u,v,dΩ) + graddiv(u,v,dΩ) + Coriolis(u,v,dΩ) 
res((u,p),(v,q),dΩ) = res_u(u,v,dΩ) - (1/ρₒ)*∫(divergence(v)*p)dΩ - (1/ρₒ)*∫(divergence(u)*q)dΩ + b((v,q),dΓ)

jac_h(x,dx,dy) = jac(x,dx,dy,dΩ)
res_h(x,dy) = res(x,dy,dΩ)
op = FEOperator(res_h,jac_h,X,Y)

solver_u = LUSolver()
solver_p = CGSolver(JacobiLinearSolver();maxiter=20,atol=1e-14,rtol=1.e-6,verbose=true)
solver_p.log.depth = 4

bblocks  = [NonlinearSystemBlock() LinearSystemBlock();
            LinearSystemBlock()    BiformBlock((p,q) -> ∫(-((ρₒ*ρₒ)/α)*p*q)dΩ,Q,Q)]
coeffs = [1.0 1.0;
          0.0 1.0]  
P = BlockTriangularSolver(bblocks,[solver_u,solver_p],coeffs,:upper)
solver = FGMRESSolver(20,P;atol=1e-11,rtol=1.e-8,verbose=true)
solver.log.depth = 2

nlsolver = NewtonSolver(solver;maxiter=20,atol=1e-10,rtol=1.e-12,verbose=true)
uh,ph = solve(nlsolver,op)

--------------- Starting Newton-Raphson solver --------
  > Iteration   0 - Residuals: 1.03e-06,   1.00e+00 
    --------------- Starting FGMRES solver ----------------
      > Iteration   0 - Residuals: 1.03e-06,   1.00e+00 
        --------------- Starting CG solver --------------------
          > Iteration   0 - Residuals: 0.00e+00,   1.00e+00 
        Solver CG finished with reason SOLVER_CONVERGED_ATOL
        Iterations:   0 - Residuals: 0.00e+00,   NaN 
        --------------- Exiting CG solver ---------------------
      > Iteration   1 - Residuals: 5.70e-08,   5.54e-02 
        --------------- Starting CG solver --------------------
          > Iteration   0 - Residuals: 1.00e+00,   1.00e+00 
          > Iteration   1 - Residuals: 1.97e-02,   1.97e-02 
          > Iteration   2 - Residuals: 6.97e-03,   6.97e-03 
          > Iteration   3 - Residuals: 1.38e-16,   1.38e-16 
        Solver CG finished with reason SOLVER_CONVERGED_RTOL
        Iterations:   3 - Residuals: 1.38e-1

MultiFieldFEFunction():
 num_fields: 2
 num_cells: 3650
 DomainStyle: ReferenceDomain()
 Triangulation: BodyFittedTriangulation()
 Triangulation id: 9758018175620246081

Post-process and tidy up.

In [57]:
mag_integral = sqrt(sum( ∫( uh ⋅ uh )dΩ ))
domain_volume = sum(∫(1)dΩ)
avg_velocity_magnitude = mag_integral / domain_volume

println("Average |u| = ", avg_velocity_magnitude, " m/s. Domain volume = ", domain_volume, " m³.")

writevtk(Ω,"ArcticBoundaryCurrentModel10",cellfields=["uh"=>uh,"ph"=>ph])

Average |u| = 3.769680157237471e-5 m/s. Domain volume = 8.11299145112307 m³.


(["ArcticBoundaryCurrentModel10.vtu"],)